<a href="https://colab.research.google.com/github/AristidesAntonioOrellanaZelaya/etl-proyecto-bi/blob/main/Notebooks/Turistas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# Cargar archivo
df_turistas = pd.read_csv("https://raw.githubusercontent.com/AristidesAntonioOrellanaZelaya/etl-proyecto-bi/refs/heads/main/Data/fact_turistas.csv")

# Verificar registros
print("Registros originales:", len(df_turistas))

df_turistas.head()

Registros originales: 20000


,id_visita,fecha,pais_origen,via_ingreso,motivo_viaje,departamento_visitado,noches_estadia,gasto_total
0,1,2024-02-01,Canada,Terrestre,Negocios,Santa Ana,8,2233
1,2,2024-12-30,Estados Unidos,Terrestre,Familiares,Sonsonate,5,408
2,3,2023-05-11,Guatemala,Aerea,Familiares,Sonsonate,7,572
3,4,2024-07-18,Canada,Aerea,Ocio,Sonsonate,2,241
4,5,2024-02-05,Estados Unidos,Terrestre,Negocios,San Salvador,10,847


**Transformar**

Convertir fecha

In [2]:
df_turistas['fecha'] = pd.to_datetime(df_turistas['fecha'])

Crear atributos de tiempo

In [3]:
df_turistas['anio'] = df_turistas['fecha'].dt.year
df_turistas['mes'] = df_turistas['fecha'].dt.month
df_turistas['dia'] = df_turistas['fecha'].dt.day
df_turistas['trimestre'] = df_turistas['fecha'].dt.quarter

Limpiar campos de texto

In [4]:
columnas_texto = [
    'pais_origen',
    'via_ingreso',
    'motivo_viaje',
    'departamento_visitado'
]

for col in columnas_texto:
    df_turistas[col] = (
        df_turistas[col]
        .astype(str)
        .str.strip()
        .str.title()
    )

Eliminar duplicados

In [5]:
duplicados = df_turistas.duplicated().sum()

print("Duplicados encontrados:", duplicados)

df_turistas.drop_duplicates(inplace=True)

Duplicados encontrados: 0


Validar métricas

In [6]:
df_turistas = df_turistas[
    (df_turistas['noches_estadia'] > 0)
]

df_turistas = df_turistas[
    (df_turistas['gasto_total'] > 0)
]

Crear KPI útil para análisis

In [7]:
df_turistas['gasto_por_noche'] = (
    df_turistas['gasto_total'] /
    df_turistas['noches_estadia']
).round(2)

Revisar valores nulos

In [8]:
print(df_turistas.isnull().sum())

id_visita                0
fecha                    0
pais_origen              0
via_ingreso              0
motivo_viaje             0
departamento_visitado    0
noches_estadia           0
gasto_total              0
anio                     0
mes                      0
dia                      0
trimestre                0
gasto_por_noche          0
dtype: int64


Estadísticas generales

In [9]:
print(df_turistas.describe())

          id_visita                       fecha  noches_estadia   gasto_total  \
count  20000.000000                       20000    20000.000000  20000.000000   
mean   10000.500000  2023-01-07 23:21:54.720000        5.990350   1313.844150   
min        1.000000         2021-01-01 00:00:00        1.000000    120.000000   
25%     5000.750000         2022-01-15 00:00:00        3.000000    717.000000   
50%    10000.500000         2023-01-16 00:00:00        6.000000   1312.000000   
75%    15000.250000         2024-01-06 00:00:00        9.000000   1914.000000   
max    20000.000000         2024-12-31 00:00:00       11.000000   2499.000000   
std     5773.647028                         NaN        3.176187    688.224707   

               anio           mes           dia     trimestre  gasto_por_noche  
count  20000.000000  20000.000000  20000.000000  20000.000000     20000.000000  
mean    2022.523950      6.477200     15.726900      2.494400       364.861548  
min     2021.000000      1.

**Carga**

In [10]:
df_turistas.to_csv(
    'dw_fact_turistas.csv',
    index=False
)

print("Archivo generado correctamente")

Archivo generado correctamente


**Verificación** final

In [11]:
print("Registros finales:", len(df_turistas))

df_turistas.info()

df_turistas.head()

Registros finales: 20000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id_visita              20000 non-null  int64         
 1   fecha                  20000 non-null  datetime64[ns]
 2   pais_origen            20000 non-null  object        
 3   via_ingreso            20000 non-null  object        
 4   motivo_viaje           20000 non-null  object        
 5   departamento_visitado  20000 non-null  object        
 6   noches_estadia         20000 non-null  int64         
 7   gasto_total            20000 non-null  int64         
 8   anio                   20000 non-null  int32         
 9   mes                    20000 non-null  int32         
 10  dia                    20000 non-null  int32         
 11  trimestre              20000 non-null  int32         
 12  gasto_por_noche        20000 non-nu

,id_visita,fecha,pais_origen,via_ingreso,motivo_viaje,departamento_visitado,noches_estadia,gasto_total,anio,mes,dia,trimestre,gasto_por_noche
0,1,2024-02-01,Canada,Terrestre,Negocios,Santa Ana,8,2233,2024,2,1,1,279.12
1,2,2024-12-30,Estados Unidos,Terrestre,Familiares,Sonsonate,5,408,2024,12,30,4,81.60
2,3,2023-05-11,Guatemala,Aerea,Familiares,Sonsonate,7,572,2023,5,11,2,81.71
3,4,2024-07-18,Canada,Aerea,Ocio,Sonsonate,2,241,2024,7,18,3,120.50
4,5,2024-02-05,Estados Unidos,Terrestre,Negocios,San Salvador,10,847,2024,2,5,1,84.70
